# Finne viktige forklaringsvariabler vha. K-fold CV

## Imports/sessions

In [121]:
from snowflake.snowpark.functions import col, sum as sum_, max as max_, datediff, round as round_, year, month, when, lit, lag, avg, count, stddev, to_date, dayofweek, weekofyear, dayofmonth, quarter, last_day
from snowflake.snowpark import Window
import re
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# from dwh_tools.get_or_create_session import get_or_create_session
from snowflake.snowpark import Session
from snowflake.snowpark.context import get_active_session
import os
import requests
import gzip
import json
# from dwh_tools.get_or_create_session import get_or_create_session
pd.set_option('display.max_columns', None)
import time
import pywin

In [122]:
def get_or_create_session(schema: str = None) -> Session:
    """
    Returns the active Snowpark session if running inside Snowflake,
    otherwise creates one locally using Azure OAuth (interactive login if needed).
 
    Parameters
    ----------
    config_path : str
        Path to a JSON config file containing Snowflake connection parameters
        (account, warehouse, role, database, schema).
 
    Returns
    -------
    Session
        A Snowpark Session object.
    """
    # If running on snowflake
    if 'POSIT_PRODUCT' in os.environ:
        session = Session.builder.getOrCreate()
        session.sql("USE DATABASE PROD_FOR_SKADE_PRODUKT_ADHOC").collect()
        if schema:
            session.sql("USE SCHEMA " + schema).collect()
        else:
            session.sql("USE SCHEMA PRODUKT_WRITE_DEV").collect()
        session.sql("USE WAREHOUSE SKADE_VWH").collect()
 
        return session
 
    try:
        session = get_active_session()
        return session
    except Exception:
        import win32api
        if schema is None:
            schema = 'PRODUKT_WRITE_DEV'
 
        connection_parameters = {
            "server": "km28161.west-europe.azure.snowflakecomputing.com",
            "warehouse": "SKADE_VWH",
            "account": "VK82539-KLP",
            "database": "PROD_FOR_SKADE_PRODUKT_ADHOC",
            "schema" : schema,
            "user": win32api.GetUserNameEx(win32api.NameUserPrincipal),  
            "authenticator": "externalbrowser"
        }
       
        # Create the session
        session = Session.builder.configs(connection_parameters).create()
        return session

In [123]:
session = get_or_create_session()

## Data

### Lese data

In [124]:
# Kundesenter data
df_inngang = session.table('elh_write.inngangsdata').to_pandas()
df_inngang.columns = df_inngang.columns.str.lower()
# TIA data
df_info = session.table('inngangsdata_info').to_pandas()
df_info.columns = df_info.columns.str.lower()
# Værdata
df_weather = pd.read_csv("../../data/oslo_weather.csv", delimiter=";")
df_weather.columns = df_weather.columns.str.lower()

### Data-fiks, inngangsdata

In [125]:
df_inngang['ankomst_dato'] = pd.to_datetime(df_inngang['ankomst_dato'], format="%d.%m.%Y %H:%M:%S").dt.date
# Respons (behandlingstid) som sekunder
df_inngang['behandlingstid'] = (
    df_inngang['behandlingstid']
    .str.split(':')
    .apply(lambda x: int(x[0]) * 60 + int(x[1]))
)
# Også etterbehandlingstid som sekunder:
df_inngang['etterbehandlingstid'] = (
    df_inngang['etterbehandlingstid']
    .str.split(':')
    .apply(lambda x: int(x[0]) * 60 + int(x[1]))
)
df_inngang["behandlet"]=df_inngang["behandlet"].eq("Behandlet")
# Aggregering til dag-nivå
df_dag = (
    df_inngang.groupby("ankomst_dato", as_index=False)
    .agg(
        antall_samtaler=("unique_id", "size"),
        behandlet_andel=("behandlet", "mean"),
        tid_i_ko_snitt=("tid_i_ko", "mean"),
        behandlingstid_snitt=("behandlingstid", "mean"),
        total_behandlingstid=("behandlingstid", "sum"),
        etterbehandligstid_snitt=("etterbehandlingstid", "mean"), 
        total_etterbehandligstid=("etterbehandlingstid", "sum")
    ).
    sort_values("ankomst_dato")
    .reset_index(drop=True)
)

### Data-fiks, TIA-data

In [126]:
# Datetime format
df_info['hf_dato'] = pd.to_datetime(df_info['hf_dato'], format="%Y-%m-%d").dt.date

### Data-fiks, værdata

In [127]:
# Fiks tid-format
df_weather["tid(norsk normaltid)"]= pd.to_datetime(df_weather["tid(norsk normaltid)"], format="%d.%m.%Y").dt.date
# Fiks datatype
df_weather["nedbør (døgn)"] = df_weather["nedbør (døgn)"].str.replace(",", ".", regex=True).astype(float)
df_weather["middeltemperatur (døgn)"] = df_weather["middeltemperatur (døgn)"].str.replace(",", ".", regex=True).astype(float)
df_weather = df_weather.rename(columns={"nedbør (døgn)": "nedbør", "middeltemperatur (døgn)": "middeltemperatur"})

### Sammenslåing av dataframes

In [128]:
df_temp = pd.merge(df_dag, df_info, left_on="ankomst_dato", right_on="hf_dato", how="left")
df = pd.merge(df_temp, df_weather, left_on="ankomst_dato", right_on="tid(norsk normaltid)", how="left")

# Temperaturavvik fra månedssnitt (per kalender-måned)
month_key = pd.to_datetime(df["ankomst_dato"]).dt.to_period("M")
df["middeltemp_mndsnitt"] = df.groupby(month_key)["middeltemperatur"].transform("mean")
df["tempavvik_fra_mndsnitt"] = df["middeltemperatur"] - df["middeltemp_mndsnitt"]

### Sammenslåing av dekninger

In [129]:
df["antall_nye_kunder_b30_ny"] = df["antall_nye_kunder_b30_mpb01_ny"] + df["antall_nye_kunder_b30_eph01_ny"] + df["antall_nye_kunder_b30_epf01_ny"] + df["antall_nye_kunder_b30_upr01_ny"]
df["antall_nye_kunder_b30_for"] = df["antall_nye_kunder_b30_mpb01_for"] + df["antall_nye_kunder_b30_eph01_for"] + df["antall_nye_kunder_b30_epf01_for"] + df["antall_nye_kunder_b30_upr01_for"]
df["antall_hf_b30_ny"] = df["antall_hf_b30_mpb01_ny"] + df["antall_hf_b30_eph01_ny"] + df["antall_hf_b30_epf01_ny"] + df["antall_hf_b30_upr01_ny"]
df["antall_hf_b30_for"] = df["antall_hf_b30_mpb01_for"] + df["antall_hf_b30_eph01_for"] + df["antall_hf_b30_epf01_for"]  + df["antall_hf_b30_upr01_for"]
df["stddev_premieendring_b30_for"] = (df["stddev_premieendring_b30_eph01_for"] + df["stddev_premieendring_b30_epf01_for"] + df["stddev_premieendring_b30_mpb01_for"] + df["stddev_premieendring_b30_upr01_for"]) / 4
df["snitt_premieendring_b30_for"] = (df["snitt_premieendring_b30_eph01_for"] + df["snitt_premieendring_b30_epf01_for"] + df["snitt_premieendring_b30_mpb01_for"] + df["snitt_premieendring_b30_upr01_for"]) / 4

### Tar totalen av ny og for også:

df["antall_nye_kunder_b30_tot"] = df["antall_nye_kunder_b30_ny"] + df["antall_nye_kunder_b30_for"]
df["antall_hf_b30_tot"] = df["antall_hf_b30_ny"] + df["antall_hf_b30_for"]

### Fjerning av unødvendige kolonner:

In [130]:
# Liste med kolonnenavn som kan fjernes fra df:
cols_to_remove = [
    "hf_dato",
    "antall_nye_kunder_b7_mpb01_ny",
    "antall_nye_kunder_f30_mpb01_ny",
    "antall_nye_kunder_f7_mpb01_ny",
    "antall_hf_b7_mpb01_ny",
    "antall_hf_f30_mpb01_ny",
    "antall_hf_f7_mpb01_ny",
    "antall_nye_kunder_b7_eph01_for",
    "antall_nye_kunder_f30_eph01_for",
    "antall_nye_kunder_f7_eph01_for",
    "antall_hf_b7_eph01_for",
    "stddev_premieendring_b7_eph01_for",
    "snitt_premieendring_b7_eph01_for",
    "antall_hf_f30_eph01_for",
    "stddev_premieendring_f30_eph01_for",
    "snitt_premieendring_f30_eph01_for",
    "antall_hf_f7_eph01_for",
    "stddev_premieendring_f7_eph01_for",
    "snitt_premieendring_f7_eph01_for",
    "antall_nye_kunder_b7_eph01_ny",
    "antall_nye_kunder_f30_eph01_ny",
    "antall_nye_kunder_f7_eph01_ny",
    "antall_hf_b7_eph01_ny",
    "antall_hf_f30_eph01_ny",
    "antall_hf_f7_eph01_ny",
    "antall_nye_kunder_b7_epf01_ny",
    "antall_nye_kunder_f30_epf01_ny",
    "antall_nye_kunder_f7_epf01_ny",
    "antall_hf_b7_epf01_ny",
    "antall_hf_f30_epf01_ny",
    "antall_hf_f7_epf01_ny",
    "antall_nye_kunder_b7_epf01_for",
    "antall_nye_kunder_f30_epf01_for",
    "antall_nye_kunder_f7_epf01_for",
    "antall_hf_b7_epf01_for",
    "stddev_premieendring_b7_epf01_for",
    "snitt_premieendring_b7_epf01_for",
    "antall_hf_f30_epf01_for",
    "stddev_premieendring_f30_epf01_for",
    "snitt_premieendring_f30_epf01_for",
    "antall_hf_f7_epf01_for",
    "stddev_premieendring_f7_epf01_for",
    "snitt_premieendring_f7_epf01_for",
    "antall_nye_kunder_b7_upr01_ny",
    "antall_nye_kunder_f30_upr01_ny",
    "antall_nye_kunder_f7_upr01_ny",
    "antall_hf_b7_upr01_ny",
    "antall_hf_f30_upr01_ny",
    "antall_hf_f7_upr01_ny",
    "antall_nye_kunder_b7_mpb01_for",
    "antall_nye_kunder_f30_mpb01_for",
    "antall_nye_kunder_f7_mpb01_for",
    "antall_hf_b7_mpb01_for",
    "stddev_premieendring_b7_mpb01_for",
    "snitt_premieendring_b7_mpb01_for",
    "antall_hf_f30_mpb01_for",
    "stddev_premieendring_f30_mpb01_for",
    "snitt_premieendring_f30_mpb01_for",
    "antall_hf_f7_mpb01_for",
    "stddev_premieendring_f7_mpb01_for",
    "snitt_premieendring_f7_mpb01_for",
    "antall_nye_kunder_b7_upr01_for",
    "antall_nye_kunder_f30_upr01_for",
    "antall_nye_kunder_f7_upr01_for",
    "antall_hf_b7_upr01_for",
    "stddev_premieendring_b7_upr01_for",
    "snitt_premieendring_b7_upr01_for",
    "antall_hf_f30_upr01_for",
    "stddev_premieendring_f30_upr01_for",
    "snitt_premieendring_f30_upr01_for",
    "antall_hf_f7_upr01_for",
    "stddev_premieendring_f7_upr01_for",
    "snitt_premieendring_f7_upr01_for",
    "navn", 
    "stasjon", 
    "tid(norsk normaltid)"
]
df = df.drop(columns=cols_to_remove)


In [131]:
df[["ankomst_dato", "er_helligdag", "er_dag_foer_helligdag", "er_dag_etter_helligdag", "ukedag"]].head(10)
print(df.columns)

Index(['ankomst_dato', 'antall_samtaler', 'behandlet_andel', 'tid_i_ko_snitt',
       'behandlingstid_snitt', 'total_behandlingstid',
       'etterbehandligstid_snitt', 'total_etterbehandligstid',
       'antall_nye_kunder_b30_mpb01_ny', 'antall_hf_b30_mpb01_ny',
       'antall_nye_kunder_b30_eph01_for', 'antall_hf_b30_eph01_for',
       'stddev_premieendring_b30_eph01_for',
       'snitt_premieendring_b30_eph01_for', 'antall_nye_kunder_b30_eph01_ny',
       'antall_hf_b30_eph01_ny', 'antall_nye_kunder_b30_epf01_ny',
       'antall_hf_b30_epf01_ny', 'antall_nye_kunder_b30_epf01_for',
       'antall_hf_b30_epf01_for', 'stddev_premieendring_b30_epf01_for',
       'snitt_premieendring_b30_epf01_for', 'antall_nye_kunder_b30_upr01_ny',
       'antall_hf_b30_upr01_ny', 'antall_nye_kunder_b30_mpb01_for',
       'antall_hf_b30_mpb01_for', 'stddev_premieendring_b30_mpb01_for',
       'snitt_premieendring_b30_mpb01_for', 'antall_nye_kunder_b30_upr01_for',
       'antall_hf_b30_upr01_for', 'stdde

## Velge variabler (CV)

In [132]:
import statsmodels.api as sm
import statsmodels.formula.api as smf


In [133]:
# Number of folds
K = 10
# Variabler inkludert fra start:
covs = ['antall_nye_kunder_b30_mpb01_ny', 'antall_hf_b30_mpb01_ny',
       'antall_nye_kunder_b30_eph01_for', 'antall_hf_b30_eph01_for',
       'stddev_premieendring_b30_eph01_for',
       'snitt_premieendring_b30_eph01_for', 'antall_nye_kunder_b30_eph01_ny',
       'antall_hf_b30_eph01_ny', 'antall_nye_kunder_b30_epf01_ny',
       'antall_hf_b30_epf01_ny', 'antall_nye_kunder_b30_epf01_for',
       'antall_hf_b30_epf01_for', 'stddev_premieendring_b30_epf01_for',
       'snitt_premieendring_b30_epf01_for', 'antall_nye_kunder_b30_upr01_ny',
       'antall_hf_b30_upr01_ny', 'antall_nye_kunder_b30_mpb01_for',
       'antall_hf_b30_mpb01_for', 'stddev_premieendring_b30_mpb01_for',
       'snitt_premieendring_b30_mpb01_for', 'antall_nye_kunder_b30_upr01_for',
       'antall_hf_b30_upr01_for', 'stddev_premieendring_b30_upr01_for',
       'snitt_premieendring_b30_upr01_for', 'aar', 'kvartal', 'maaned',
       'ukenummer', 'ukedag', 'dag_i_maaned', 'er_helg', 'helligdag',
       'er_helligdag', 'er_dag_foer_helligdag', 'er_dag_etter_helligdag',
       'middeltemperatur', 'tempavvik_fra_mndsnitt', 'nedbør', 'antall_nye_kunder_b30_ny',
       'antall_nye_kunder_b30_for', 'antall_hf_b30_ny', 'antall_hf_b30_for',
       'stddev_premieendring_b30_for', 'snitt_premieendring_b30_for',
       'antall_nye_kunder_b30_tot', 'antall_hf_b30_tot']

covs = ['aar', 'kvartal', 'maaned', 'ukenummer', 'ukedag', 'dag_i_maaned', 'er_helg', 'helligdag', 'er_helligdag', 'er_dag_foer_helligdag', 'er_dag_etter_helligdag',
       'middeltemperatur', 'tempavvik_fra_mndsnitt', 'nedbør', 'antall_nye_kunder_b30_ny',
       'antall_nye_kunder_b30_for', 'antall_hf_b30_ny', 'antall_hf_b30_for',
       'stddev_premieendring_b30_for', 'snitt_premieendring_b30_for',
       'antall_nye_kunder_b30_tot', 'antall_hf_b30_tot']

# covs = ['tempavvik_fra_mndsnitt', 'nedbør', 'antall_nye_kunder_b30_ny',
#        'antall_nye_kunder_b30_for', 'antall_hf_b30_ny', 'antall_hf_b30_for',
#        'stddev_premieendring_b30_for', 'snitt_premieendring_b30_for',
#        'antall_nye_kunder_b30_tot', 'antall_hf_b30_tot']

covs = ['kvartal', 'maaned', 'ukenummer', 'ukedag', 'dag_i_maaned', 'er_helg', 'helligdag', 'er_helligdag', 'er_dag_foer_helligdag', 'er_dag_etter_helligdag',
       'middeltemperatur', 'tempavvik_fra_mndsnitt', 'nedbør', 'antall_nye_kunder_b30_ny',
       'antall_nye_kunder_b30_for', 'antall_hf_b30_ny', 'antall_hf_b30_for',
       'stddev_premieendring_b30_for', 'snitt_premieendring_b30_for',
       'antall_nye_kunder_b30_tot', 'antall_hf_b30_tot']

In [134]:
from sklearn import linear_model
from sklearn import preprocessing
from sklearn import model_selection
from sklearn.metrics import mean_tweedie_deviance, make_scorer

In [135]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_poisson_deviance, make_scorer

DEFAULT_CATEGORICAL_FEATURES = [
    "aar", "kvartal", "maaned", "ukenummer", "ukedag", "dag_i_maaned",
    "er_helg", "helligdag", "er_helligdag",
    "er_dag_foer_helligdag", "er_dag_etter_helligdag",
]

def _build_poisson_pipeline(
    X: pd.DataFrame,
    categorical_features: list | None = None,
) -> Pipeline:
    # Tving one-hot for eksplisitte kategoriske variabler, selv om de er lagret som int.
    forced_cat = set(categorical_features or []) & set(X.columns)
    inferred_cat = {
        c for c in X.columns
        if pd.api.types.is_object_dtype(X[c])
        or isinstance(X[c].dtype, pd.CategoricalDtype)
        or pd.api.types.is_bool_dtype(X[c])
    }
    cat_cols = sorted(forced_cat | inferred_cat)
    num_cols = [c for c in X.columns if c not in cat_cols]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler(with_mean=False))
                ]),
                num_cols
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))
                ]),
                cat_cols
            )
        ],
        remainder="drop"
    )

    model = PoissonRegressor(alpha=0.0, max_iter=3000)
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])
    return pipe


def forward_stagewise_cv(
    df: pd.DataFrame,
    response: str,
    candidate_covs: list,
    k: int = 10,
    max_steps: int = None,
    random_state: int = 42,
    tol: float = 1e-4,
    verbose: bool = True,
    categorical_features: list | None = None,
):
    remaining = [c for c in candidate_covs if c in df.columns and c != response]
    if len(remaining) == 0:
        raise ValueError("Ingen gyldige candidate_covs finnes i df.")

    y = df[response].copy()
    mask = y.notna()
    y = y.loc[mask]

    selected = []
    history_rows = []
    per_step_tables = {}

    if max_steps is None:
        max_steps = len(remaining)
    max_steps = min(max_steps, len(remaining))

    cv = KFold(n_splits=k, shuffle=True, random_state=random_state)
    scoring = {
        "poisson_dev": make_scorer(mean_poisson_deviance, greater_is_better=False),
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    }

    best_dev_so_far = np.inf

    for step in range(1, max_steps + 1):
        if verbose:
            print(f"\n=== Steg {step} ===")
            print(f"Allerede valgt: {selected if selected else 'Ingen'}")

        step_results = []

        for cov in remaining:
            current_covs = selected + [cov]
            X = df.loc[mask, current_covs].copy()
            pipe = _build_poisson_pipeline(X, categorical_features=categorical_features)

            cv_res = cross_validate(
                pipe,
                X,
                y,
                cv=cv,
                scoring=scoring,
                n_jobs=-1,
                error_score="raise"
            )

            mean_dev = -cv_res["test_poisson_dev"].mean()
            std_dev = cv_res["test_poisson_dev"].std()
            mean_mae = -cv_res["test_mae"].mean()
            mean_r2 = cv_res["test_r2"].mean()

            step_results.append({
                "candidate": cov,
                "n_covariates": len(current_covs),
                "cv_poisson_deviance_mean": mean_dev,
                "cv_poisson_deviance_std": std_dev,
                "cv_mae_mean": mean_mae,
                "cv_r2_mean": mean_r2
            })

            if verbose:
                print(
                    f"  Kandidat: {cov:<35} | "
                    f"Poisson deviance: {mean_dev:10.4f} | "
                    f"MAE: {mean_mae:8.4f} | R2: {mean_r2:8.4f}"
                )

        step_df = pd.DataFrame(step_results).sort_values(
            by="cv_poisson_deviance_mean", ascending=True
        ).reset_index(drop=True)
        per_step_tables[step] = step_df

        best_row = step_df.iloc[0]
        best_cov = best_row["candidate"]
        best_dev = best_row["cv_poisson_deviance_mean"]

        improvement = best_dev_so_far - best_dev
        keep_going = improvement > tol

        history_rows.append({
            "step": step,
            "added_covariate": best_cov,
            "cv_poisson_deviance_mean": best_dev,
            "cv_mae_mean": best_row["cv_mae_mean"],
            "cv_r2_mean": best_row["cv_r2_mean"],
            "improvement_vs_prev": improvement if np.isfinite(best_dev_so_far) else np.nan,
            "selected_covariates": selected + [best_cov]
        })

        if verbose:
            print("-" * 90)
            print(
                f"Valgt i steg {step}: {best_cov} | "
                f"Ny beste deviance: {best_dev:.4f} | "
                f"Forbedring: {improvement if np.isfinite(improvement) else np.nan:.6f}"
            )

        if not keep_going and np.isfinite(best_dev_so_far):
            if verbose:
                print(f"Stopper: forbedring ({improvement:.6f}) <= tol ({tol}).")
            break

        selected.append(best_cov)
        remaining.remove(best_cov)
        best_dev_so_far = best_dev

        if len(remaining) == 0:
            break

    history = pd.DataFrame(history_rows)
    return selected, history, per_step_tables


# Kjoring
selected_covs, history, per_step = forward_stagewise_cv(
    df=df,
    response="antall_samtaler",
    candidate_covs=covs,
    k=K,
    max_steps=15,
    random_state=42,
    tol=1e-4,
    verbose=True,
    categorical_features=DEFAULT_CATEGORICAL_FEATURES,
)

print("\nFerdig. Valgte kovariater i rekkefolge:")
print(selected_covs)

print("\nHistorikk:")
display(history)


=== Steg 1 ===
Allerede valgt: Ingen
  Kandidat: kvartal                             | Poisson deviance:    33.9605 | MAE:  79.4014 | R2:   0.0157
  Kandidat: maaned                              | Poisson deviance:    29.1750 | MAE:  70.7427 | R2:   0.1576
  Kandidat: ukenummer                           | Poisson deviance:    29.4316 | MAE:  71.6201 | R2:   0.1409
  Kandidat: ukedag                              | Poisson deviance:    35.9989 | MAE:  81.9641 | R2:  -0.0422
  Kandidat: dag_i_maaned                        | Poisson deviance:    38.6900 | MAE:  85.1320 | R2:  -0.1318
  Kandidat: er_helg                             | Poisson deviance:    35.9844 | MAE:  81.8300 | R2:  -0.0427
  Kandidat: helligdag                           | Poisson deviance:    35.7432 | MAE:  81.5091 | R2:  -0.0331
  Kandidat: er_helligdag                        | Poisson deviance:    35.6541 | MAE:  81.3164 | R2:  -0.0301
  Kandidat: er_dag_foer_helligdag               | Poisson deviance:    35.9989 | M

,step,added_covariate,cv_poisson_deviance_mean,cv_mae_mean,cv_r2_mean,improvement_vs_prev,selected_covariates
0,1,maaned,29.175007,70.742656,0.157610,NaN,[maaned]
1,2,stddev_premieendring_b30_for,27.859374,69.036849,0.190711,1.315632,"[maaned, stddev_premieendring_b30_for]"
2,3,antall_hf_b30_ny,27.000470,67.094700,0.220871,0.858905,"[maaned, stddev_premieendring_b30_for, antall_..."
3,4,tempavvik_fra_mndsnitt,26.706251,66.874325,0.230407,0.294219,"[maaned, stddev_premieendring_b30_for, antall_..."
4,5,middeltemperatur,26.458701,67.078466,0.234289,0.247550,"[maaned, stddev_premieendring_b30_for, antall_..."
5,6,er_dag_foer_helligdag,26.213688,66.885432,0.241432,0.245013,"[maaned, stddev_premieendring_b30_for, antall_..."
6,7,ukedag,26.119498,66.924640,0.244813,0.094191,"[maaned, stddev_premieendring_b30_for, antall_..."
7,8,antall_hf_b30_tot,26.063212,66.938068,0.247188,0.056286,"[maaned, stddev_premieendring_b30_for, antall_..."
8,9,kvartal,26.058901,66.935921,0.247293,0.004310,"[maaned, stddev_premieendring_b30_for, antall_..."
9,10,antall_nye_kunder_b30_for,26.083797,66.971707,0.246470,-0.024895,"[maaned, stddev_premieendring_b30_for, antall_..."


In [136]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_poisson_deviance, make_scorer
from sklearn.model_selection import KFold, cross_validate

def evaluate_cv_metrics(
    df: pd.DataFrame,
    response: str,
    features: list,
    k: int = 10,
    random_state: int = 42,
    categorical_features: list | None = None,
):
    y = df[response].copy()
    mask = y.notna()
    y = y.loc[mask]

    cv = KFold(n_splits=k, shuffle=True, random_state=random_state)
    scoring = {
        "poisson_dev": make_scorer(mean_poisson_deviance, greater_is_better=False),
        "mae": "neg_mean_absolute_error",
        "r2": "r2",
    }

    if len(features) == 0:
        X = pd.DataFrame(index=y.index)
        model = DummyRegressor(strategy="mean")
    else:
        X = df.loc[mask, features].copy()
        model = _build_poisson_pipeline(X, categorical_features=categorical_features)

    cv_res = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        error_score="raise",
    )

    return {
        "poisson_dev_mean": -cv_res["test_poisson_dev"].mean(),
        "poisson_dev_std": cv_res["test_poisson_dev"].std(),
        "mae_mean": -cv_res["test_mae"].mean(),
        "r2_mean": cv_res["test_r2"].mean(),
    }

# Evaluer valgt modell vs nullmodell (kun gjennomsnitt)
best_features = selected_covs.copy()
best_metrics = evaluate_cv_metrics(
    df=df,
    response="antall_samtaler",
    features=best_features,
    k=K,
    random_state=42,
    categorical_features=DEFAULT_CATEGORICAL_FEATURES,
 )
null_metrics = evaluate_cv_metrics(
    df=df,
    response="antall_samtaler",
    features=[],
    k=K,
    random_state=42,
 )

# Skill score basert på Poisson deviance: >0 er bedre enn nullmodell, 1 er perfekt
poisson_skill = 1 - (best_metrics["poisson_dev_mean"] / null_metrics["poisson_dev_mean"])

print("Valgte kovariater:", best_features)
print("\nCV-metrikker (valgt modell):")
print(best_metrics)
print("\nCV-metrikker (nullmodell):")
print(null_metrics)
print(f"\nPoisson skill score vs nullmodell: {poisson_skill:.4f}")

Valgte kovariater: ['maaned', 'stddev_premieendring_b30_for', 'antall_hf_b30_ny', 'tempavvik_fra_mndsnitt', 'middeltemperatur', 'er_dag_foer_helligdag', 'ukedag', 'antall_hf_b30_tot', 'kvartal']

CV-metrikker (valgt modell):
{'poisson_dev_mean': np.float64(26.058901473864722), 'poisson_dev_std': np.float64(8.197153535805754), 'mae_mean': np.float64(66.93592121730971), 'r2_mean': np.float64(0.24729297222841273)}

CV-metrikker (nullmodell):
{'poisson_dev_mean': np.float64(35.99236811924217), 'poisson_dev_std': np.float64(8.044325988323733), 'mae_mean': np.float64(81.93440000000001), 'r2_mean': np.float64(-0.042851644267455094)}

Poisson skill score vs nullmodell: 0.2760
